# Ajuste parametrico — Pozo C147 (Mark II M-912D-365-168)

**Datos firmes** (del JSON): geometria API 11E, sarta, bomba, motor HP/RPM.

**Datos a ajustar**: N_total (transmision), M_cw/L_cw/tau_cw (contrapeso), c_friction, J_b (inercia).

**Referencia de campo**: F_min=7,580 lbs, F_max=27,014 lbs, stroke=165.6"

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from aib_simulator.well_loader import load_well, add_estimates
from aib_simulator.coupling import Simulator
from aib_simulator.geometry import (
    full_kinematics_mark2, mechanism_positions_mark2
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# 1. Cargar datos firmes del JSON
p = load_well('../M-912D-365-168.json', well_key='well_C147')
p = add_estimates(p)

# Datos de referencia del campo
F_min_field = 7580.53 * 4.4482   # 33,717 N
F_max_field = 27014.31 * 4.4482  # 120,170 N
S_field = 165.6 * 0.0254         # 4.206 m

print(f"Pozo C147 — {p['unit_model']} (Mark II)")
print(f"Sarta: {p['rod_L']:.0f}m, Bomba: {p['L_bomba']}m, Motor: {p['motor_HP']}HP")
print(f"Campo: F=[{F_min_field/1e3:.1f}, {F_max_field/1e3:.1f}] kN, S={S_field:.3f}m")

## Verificacion de cinematica Mark II

In [ ]:
# Cinematica Mark II con parametros reales
theta = np.linspace(0, 2*np.pi, 7200)
kin = full_kinematics_mark2(theta, 1.0,
    p['A_beam'], p['P_rocker'], p['C_pitman'],
    p['I_geom'], p['H_offset'], p['R_crank'])

stroke_calc = (np.max(kin['y_PR']) - np.min(kin['y_PR'])) / 0.0254
TF_max = np.max(np.abs(kin['TF'])) / 0.0254

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mecanismo en 12 posiciones
ax = axes[0]
for deg in range(0, 360, 30):
    pos = mechanism_positions_mark2(np.radians(deg),
        p['A_beam'], p['P_rocker'], p['C_pitman'],
        p['I_geom'], p['H_offset'], p['R_crank'], p['G_height'])
    cc, cp, sb, eb, hh = pos['crank_center'], pos['crank_pin'], pos['saddle_bearing'], pos['equalizer'], pos['horsehead']
    ax.plot([cc[0], cp[0]], [cc[1], cp[1]], 'r-', lw=1.5, alpha=0.4)
    ax.plot([cp[0], eb[0]], [cp[1], eb[1]], color='orange', lw=1, alpha=0.4)
    ax.plot([sb[0], hh[0]], [sb[1], hh[1]], 'b-', lw=2, alpha=0.4)
    ax.plot([sb[0], eb[0]], [sb[1], eb[1]], 'g-', lw=2, alpha=0.4)
ax.plot(*pos['saddle_bearing'], 'ks', ms=8)
ax.plot(*pos['crank_center'], 'k^', ms=8)
ax.set_aspect('equal')
ax.set_title('Mecanismo Mark II M-912D-365-168')
ax.grid(True, alpha=0.2)

# y_PR vs theta
ax = axes[1]
ax.plot(np.degrees(theta), kin['y_PR'], 'b-', lw=2)
ax.set_xlabel('theta [deg]'); ax.set_ylabel('y_PR [m]')
ax.set_title(f'Carrera: {stroke_calc:.1f} in (campo: 165.6)')
ax.grid(True, alpha=0.3); ax.set_xlim(0, 360)

# TF vs theta
ax = axes[2]
ax.plot(np.degrees(theta), kin['TF'] / 0.0254, 'purple', lw=2)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('theta [deg]'); ax.set_ylabel('TF [in/rad]')
ax.set_title(f'TF max: {TF_max:.1f} in/rad (catalogo: 75.2)')
ax.grid(True, alpha=0.3); ax.set_xlim(0, 360)

plt.suptitle('Cinematica Mark II — Verificacion contra catalogo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Simulacion y funcion de costo

Se define una funcion que corre la simulacion con parametros dados y retorna el error respecto a los datos de campo (PPRL, MPRL, SPM).

In [ ]:
def run_and_evaluate(params, n_cycles=8, target_SPM=6.0):
    """Corre la simulacion y retorna metricas del ultimo ciclo.
    
    Returns: dict con PPRL, MPRL, SPM, omega_var, y_PR, F_PR del ultimo ciclo.
             None si la simulacion diverge.
    """
    try:
        sim = Simulator(params, motor_model='linear')
        sim.initialize(theta_0=0.0, omega_0_crank=2*np.pi*target_SPM/60)
        
        n_sc = int(60 / target_SPM / params['dt'])
        n_tot = n_cycles * n_sc
        
        omegas = np.zeros(n_tot)
        F_PRs = np.zeros(n_tot)
        y_PRs = np.zeros(n_tot)
        thetas = np.zeros(n_tot)
        
        for step in range(n_tot):
            state = sim.step()
            omegas[step] = state['omega']
            F_PRs[step] = state['F_PR']
            y_PRs[step] = state['y_PR']
            thetas[step] = state['theta']
            
            # Detectar divergencia
            if abs(omegas[step]) > 50 or np.isnan(omegas[step]):
                return None
        
        # Ultimo ciclo
        theta_mod = thetas % (2*np.pi)
        cx = np.where(np.diff(theta_mod) < -np.pi)[0]
        if len(cx) >= 2:
            i0, i1 = cx[-2], cx[-1]
        else:
            i0, i1 = n_tot - n_sc, n_tot - 1
        
        om = omegas[i0:i1]
        F = F_PRs[i0:i1]
        y = y_PRs[i0:i1]
        
        om_mean = np.mean(om)
        if om_mean <= 0:
            return None
        
        return {
            'PPRL': np.max(F),
            'MPRL': np.min(F),
            'SPM': om_mean * 60 / (2*np.pi),
            'omega_var': (np.max(om) - np.min(om)) / om_mean * 100,
            'y_PR': y,
            'F_PR': F,
        }
    except Exception:
        return None


def cost_function(x, base_params, target_SPM=6.0):
    """Funcion de costo para el optimizador.
    
    x = [N_total, M_cw, L_cw, tau_cw_deg, c_friction, J_b]
    
    Minimiza el error en PPRL, MPRL y SPM respecto a datos de campo.
    """
    N_total, M_cw, L_cw, tau_cw_deg, c_friction, J_b = x
    
    params = base_params.copy()
    params['N_total'] = N_total
    params['M_cw'] = M_cw
    params['L_cw'] = L_cw
    params['tau_cw'] = np.radians(tau_cw_deg)
    params['c_friction'] = c_friction
    params['J_b'] = J_b
    
    result = run_and_evaluate(params, n_cycles=8, target_SPM=target_SPM)
    
    if result is None:
        return 1e6  # penalizar divergencia
    
    # Errores normalizados
    err_pprl = ((result['PPRL'] - F_max_field) / F_max_field) ** 2
    err_mprl = ((result['MPRL'] - F_min_field) / F_min_field) ** 2
    err_spm = ((result['SPM'] - target_SPM) / target_SPM) ** 2
    # Penalizar variacion de omega excesiva
    err_var = max(0, result['omega_var'] - 20) ** 2 / 400
    
    cost = err_pprl + err_mprl + err_spm + 0.1 * err_var
    return cost

print("Funciones de simulacion y costo definidas.")

## Optimizacion con differential_evolution

Busca los 6 parametros desconocidos que minimizan el error entre la simulacion y los datos de campo.

In [ ]:
from scipy.optimize import differential_evolution

# Rangos de busqueda: [N_total, M_cw, L_cw, tau_cw_deg, c_friction, J_b]
bounds = [
    (100, 300),      # N_total (relacion de transmision)
    (3000, 10000),   # M_cw (masa contrapeso, kg)
    (0.5, 2.0),      # L_cw (brazo contrapeso, m)
    (0, 180),        # tau_cw (fase contrapeso, grados)
    (1, 500),        # c_friction (N*m*s/rad)
    (1000, 8000),    # J_b (inercia balancin, kg*m²)
]

param_names = ['N_total', 'M_cw', 'L_cw', 'tau_cw_deg', 'c_friction', 'J_b']

print("Iniciando optimizacion (puede tomar varios minutos)...")
print(f"Target: PPRL={F_max_field/1e3:.1f}kN, MPRL={F_min_field/1e3:.1f}kN, SPM=6.0")
print()

t0 = time.time()
result = differential_evolution(
    cost_function, bounds, args=(p, 6.0),
    maxiter=30, popsize=10, seed=42, tol=0.005,
    disp=True, workers=1
)
elapsed = time.time() - t0

print(f"\nOptimizacion completada en {elapsed:.0f}s")
print(f"Costo final: {result.fun:.6f}")
print(f"\nParametros optimos:")
for name, val in zip(param_names, result.x):
    print(f"  {name:>15} = {val:.2f}")

## Resultados: simulacion con parametros ajustados vs campo

In [ ]:
# Aplicar parametros optimos y correr simulacion final
N_total_opt, M_cw_opt, L_cw_opt, tau_opt, cf_opt, Jb_opt = result.x

p_opt = p.copy()
p_opt['N_total'] = N_total_opt
p_opt['M_cw'] = M_cw_opt
p_opt['L_cw'] = L_cw_opt
p_opt['tau_cw'] = np.radians(tau_opt)
p_opt['c_friction'] = cf_opt
p_opt['J_b'] = Jb_opt

res_opt = run_and_evaluate(p_opt, n_cycles=12, target_SPM=6.0)

# Dashboard comparativo
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Carta de superficie simulada
ax = axes[0]
y_norm = res_opt['y_PR'] - np.min(res_opt['y_PR'])
ax.plot(y_norm, res_opt['F_PR'] / 1e3, 'b-', linewidth=1.5, label='Simulacion')
ax.axhline(F_max_field / 1e3, color='r', ls='--', alpha=0.5, label=f'PPRL campo: {F_max_field/1e3:.1f} kN')
ax.axhline(F_min_field / 1e3, color='g', ls='--', alpha=0.5, label=f'MPRL campo: {F_min_field/1e3:.1f} kN')
ax.set_xlabel('Posicion [m]')
ax.set_ylabel('F_PR [kN]')
ax.set_title('Carta de Superficie — Simulada vs Campo')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Comparacion de metricas
ax = axes[1]
ax.axis('off')
metrics = [
    ['Parametro', 'Campo', 'Simulado', 'Error'],
    ['PPRL [kN]', f'{F_max_field/1e3:.1f}', f'{res_opt["PPRL"]/1e3:.1f}',
     f'{abs(res_opt["PPRL"]-F_max_field)/F_max_field*100:.1f}%'],
    ['MPRL [kN]', f'{F_min_field/1e3:.1f}', f'{res_opt["MPRL"]/1e3:.1f}',
     f'{abs(res_opt["MPRL"]-F_min_field)/F_min_field*100:.1f}%'],
    ['SPM', '6.0', f'{res_opt["SPM"]:.1f}',
     f'{abs(res_opt["SPM"]-6)/6*100:.1f}%'],
    ['Var omega [%]', '5-15', f'{res_opt["omega_var"]:.1f}', ''],
    ['Stroke [m]', f'{S_field:.3f}', f'{np.max(y_norm):.3f}', ''],
]
table = ax.table(cellText=metrics, loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.5)
for i in range(5):
    table[0, i].set_facecolor('#4472C4')
    table[0, i].set_text_props(color='white', fontweight='bold')
ax.set_title('Comparacion campo vs simulacion', fontsize=13, fontweight='bold', pad=20)

# 3. Parametros ajustados
ax = axes[2]
ax.axis('off')
fitted = [
    ['Parametro', 'Valor ajustado'],
    ['N_total', f'{N_total_opt:.1f}'],
    ['M_cw [kg]', f'{M_cw_opt:.0f}'],
    ['L_cw [m]', f'{L_cw_opt:.2f}'],
    ['tau_cw [deg]', f'{tau_opt:.1f}'],
    ['c_friction [N*m*s/rad]', f'{cf_opt:.1f}'],
    ['J_b [kg*m²]', f'{Jb_opt:.0f}'],
]
table2 = ax.table(cellText=fitted, loc='center', cellLoc='center')
table2.auto_set_font_size(False); table2.set_fontsize(11); table2.scale(1.2, 1.5)
for i in range(2):
    table2[0, i].set_facecolor('#70AD47')
    table2[0, i].set_text_props(color='white', fontweight='bold')
ax.set_title('Parametros ajustados', fontsize=13, fontweight='bold', pad=20)

plt.suptitle(f'Ajuste parametrico — Pozo C147 (Mark II M-912D-365-168)',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()